# 04 GAT readiness model on Junyi (C1 Adaptive Curriculum Engine)

Trains the Graph Attention Network that scores how ready a learner is for a concept, from their BKT mastery of the concept and of its prerequisites. The trained model is exported as `gat-v1.npz` and runs with numpy inside `curriculum-service`.

**Task.** For every answer *t*: predict whether it is correct, from features computed with answers before *t* only.

**Graph-agnostic features.** Each node (exercise) gets 5 numbers that exist in any prerequisite graph, so a model trained on Junyi can score the Java PF + DSA graph:

| Feature | Meaning |
|---|---|
| `mastery` | BKT probability the node is learned (notebook 03 parameters) |
| `log_evidence` | log(1 + answers on this node so far) |
| `log_depth` | log(1 + longest prerequisite chain below the node) |
| `log_prereq_count` | log(1 + number of prerequisites) |
| `mean_prereq_mastery` | mean mastery of its prerequisites (1.0 when it has none) |

**Model.** One GAT layer over the target node and its direct prerequisites (with a self-loop), multi-head attention, then a small classifier. The attention weights over the prerequisite edges feed the explanation: weak prerequisite = argmax of attention x (1 - mastery).

**Evaluation.** 5-fold cross-validation over the training students (GAT vs BKT-only on the same folds, paired t-test), then one final model on all training students, scored on the held-out test students.

Needs notebooks 01, 02 and 03. Use a **GPU runtime** (T4): Runtime > Change runtime type.

In [ ]:
# Where outputs are stored: "drive", "runtime" or "local" (see notebook 01).
STORAGE = "drive"
# True: 3,000 training and 1,000 test students, 1 epoch, for a fast check.
QUICK = False

In [ ]:
import sys
!{sys.executable} -m pip install -q pyarrow scikit-learn networkx scipy

from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if not ON_COLAB:
    STORAGE = "local"

if STORAGE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
    except Exception as exc:
        print(f"Drive could not be mounted ({type(exc).__name__}: {exc}). Using runtime storage.")
        STORAGE = "runtime"
if STORAGE == "runtime":
    BASE = Path("/content/adaptlearn-c1")
elif STORAGE == "local":
    BASE = Path("../data").resolve()

PROCESSED = BASE / "junyi_processed"
OUT_DIR = PROCESSED / "gat"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert (PROCESSED / "interactions.parquet").exists(), "Run notebook 01 first (or upload its outputs)."
print("Storage:", STORAGE, "->", BASE)

## 1. Load answers, splits, the Junyi graph and the BKT parameters

In [ ]:
import json
import numpy as np
import pandas as pd
import networkx as nx
import torch

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CONFIG = {"SEED": SEED, "QUICK": QUICK, "HEADS": 4, "HIDDEN": 16, "CLASSIFIER": 32,
          "EPOCHS": 1 if QUICK else 3, "BATCH": 4096, "LR": 3e-3, "WEIGHT_DECAY": 1e-5,
          "LEAKY_SLOPE": 0.2, "MODEL_VERSION": "gat-v1"}

inter = pd.read_parquet(PROCESSED / "interactions.parquet",
                        columns=["user_id", "exercise_idx", "step", "correct"])
inter = inter.sort_values(["user_id", "step"], kind="mergesort").reset_index(drop=True)
splits = json.loads((PROCESSED / "splits.json").read_text())
exercise_index = json.loads((PROCESSED / "exercise_index.json").read_text())
edge_index = np.load(PROCESSED / "graph" / "edge_index.npy")  # row 0 prerequisite, row 1 exercise
bkt = pd.read_csv(PROCESSED / "bkt" / "params.csv").set_index("exercise")

fold_of = {u: int(k) for k, users in splits["folds"].items() for u in users}
test_ids = set(splits["test"])
if QUICK:
    keep = set(rng.choice(sorted(fold_of), 3_000, replace=False)) | set(rng.choice(sorted(test_ids), 1_000, replace=False))
    inter = inter[inter.user_id.isin(keep)].reset_index(drop=True)
print(f"{inter.user_id.nunique():,} students, {len(inter):,} answers")

In [ ]:
n_ex = len(exercise_index)
names = sorted(exercise_index, key=exercise_index.get)
prior = bkt.loc[names, "p_init"].to_numpy()
learn = bkt.loc[names, "p_learn"].to_numpy()
guess = bkt.loc[names, "p_guess"].to_numpy()
slip = bkt.loc[names, "p_slip"].to_numpy()

G = nx.DiGraph()
G.add_nodes_from(range(n_ex))
G.add_edges_from(zip(edge_index[0].tolist(), edge_index[1].tolist()))
assert nx.is_directed_acyclic_graph(G)
depth = np.zeros(n_ex)
for v in nx.topological_sort(G):
    preds = list(G.predecessors(v))
    depth[v] = 1 + max(depth[p] for p in preds) if preds else 0
prereqs = [sorted(G.predecessors(v)) for v in range(n_ex)]
K = max(len(p) for p in prereqs)
print(f"max prerequisites per exercise K = {K}, longest chain = {int(depth.max())}")
pd.Series([len(p) for p in prereqs]).value_counts().sort_index()

## 2. Features from answers before *t* only

One pass over every student's answers in time order. At answer *t* the features are read first, then the answer updates BKT, so nothing from *t* or later leaks into the features for *t*.

In [ ]:
import math
from tqdm.auto import tqdm

FEATURES = ["mastery", "log_evidence", "log_depth", "log_prereq_count", "mean_prereq_mastery"]
F = len(FEATURES)
# Plain Python lists: much faster than numpy for one value at a time.
P_INIT, P_LEARN, P_GUESS, P_SLIP = (a.tolist() for a in (prior, learn, guess, slip))
LOG_DEPTH = np.log1p(depth).tolist()
LOG_PREREQS = [math.log1p(len(p)) for p in prereqs]

def node_features(v, state, count):
    ps = prereqs[v]
    mean_pm = sum(state[p] for p in ps) / len(ps) if ps else 1.0
    return (state[v], math.log1p(count[v]), LOG_DEPTH[v], LOG_PREREQS[v], mean_pm)

n = len(inter)
X_t = np.zeros((n, F), np.float32)
X_n = np.zeros((n, K, F), np.float32)
M = np.zeros((n, K), bool)
p_bkt = np.zeros(n, np.float32)

users = inter.user_id.to_numpy()
ex = inter.exercise_idx.to_numpy().tolist()
y = inter.correct.to_numpy().astype(bool)
y_list = y.tolist()
new_student = np.r_[True, users[1:] != users[:-1]].tolist()

for i in tqdm(range(n), mininterval=5):
    if new_student[i]:
        state, count = list(P_INIT), [0] * n_ex
    v = ex[i]
    X_t[i] = node_features(v, state, count)
    for j, u in enumerate(prereqs[v]):
        X_n[i, j] = node_features(u, state, count)
        M[i, j] = True
    m, g, s = state[v], P_GUESS[v], P_SLIP[v]
    p_bkt[i] = m * (1 - s) + (1 - m) * g
    if y_list[i]:
        post = m * (1 - s) / (m * (1 - s) + (1 - m) * g)
    else:
        post = m * s / (m * s + (1 - m) * (1 - g))
    state[v] = post + (1 - post) * P_LEARN[v]
    count[v] += 1

fold = inter.user_id.map(fold_of).fillna(-1).astype(int).to_numpy()  # -1 = test student
is_test = np.isin(users, list(test_ids))
assert not (is_test & (fold >= 0)).any()
pd.DataFrame(X_t, columns=FEATURES).describe().round(3)

## 3. The model

In [ ]:
import torch.nn as nn
import torch.nn.functional as fn

class PrereqGAT(nn.Module):
    """One GAT layer over a node and its direct prerequisites (plus a self-loop), then a classifier."""

    def __init__(self, f, heads, hidden, classifier, slope):
        super().__init__()
        self.heads, self.hidden, self.slope = heads, hidden, slope
        self.W = nn.Linear(f, heads * hidden, bias=False)
        self.a_dst = nn.Parameter(torch.empty(heads, hidden))
        self.a_src = nn.Parameter(torch.empty(heads, hidden))
        nn.init.xavier_uniform_(self.a_dst)
        nn.init.xavier_uniform_(self.a_src)
        self.fc1 = nn.Linear(heads * hidden + f, classifier)
        self.fc2 = nn.Linear(classifier, 1)

    def forward(self, x_t, x_n, mask):
        B, k, _ = x_n.shape
        H, D = self.heads, self.hidden
        nodes = torch.cat([x_t.unsqueeze(1), x_n], 1)                    # [B, 1+K, F], slot 0 = self
        h = self.W(nodes).view(B, 1 + k, H, D)
        present = torch.cat([torch.ones(B, 1, dtype=torch.bool, device=mask.device), mask], 1)
        score = (h[:, :1] * self.a_dst).sum(-1) + (h * self.a_src).sum(-1)  # [B, 1+K, H]
        score = fn.leaky_relu(score, self.slope).masked_fill(~present.unsqueeze(-1), float("-inf"))
        alpha = torch.softmax(score, dim=1)
        z = fn.elu((alpha.unsqueeze(-1) * h).sum(1)).reshape(B, H * D)
        logit = self.fc2(torch.relu(self.fc1(torch.cat([z, x_t], -1)))).squeeze(-1)
        return logit, alpha

def new_model():
    return PrereqGAT(F, CONFIG["HEADS"], CONFIG["HIDDEN"], CONFIG["CLASSIFIER"], CONFIG["LEAKY_SLOPE"]).to(DEVICE)

print(sum(p.numel() for p in new_model().parameters()), "parameters")

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

T_xt = torch.from_numpy(X_t).to(DEVICE)
T_xn = torch.from_numpy(X_n).to(DEVICE)
T_m = torch.from_numpy(M).to(DEVICE)
T_y = torch.from_numpy(y.astype(np.float32)).to(DEVICE)

def train(rows, seed=SEED):
    torch.manual_seed(seed)
    model = new_model()
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
    rows = torch.from_numpy(rows).to(DEVICE)
    g = torch.Generator(device="cpu").manual_seed(seed)
    for epoch in range(CONFIG["EPOCHS"]):
        model.train()
        order = rows[torch.randperm(len(rows), generator=g).to(DEVICE)]
        total = 0.0
        for b in range(0, len(order), CONFIG["BATCH"]):
            idx = order[b:b + CONFIG["BATCH"]]
            logit, _ = model(T_xt[idx], T_xn[idx], T_m[idx])
            loss = fn.binary_cross_entropy_with_logits(logit, T_y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        print(f"  epoch {epoch + 1}: train loss {total / len(order):.4f}")
    return model

@torch.no_grad()
def predict(model, rows):
    model.eval()
    out = []
    for b in range(0, len(rows), 65536):
        idx = torch.from_numpy(rows[b:b + 65536]).to(DEVICE)
        out.append(torch.sigmoid(model(T_xt[idx], T_xn[idx], T_m[idx])[0]).cpu().numpy())
    return np.concatenate(out)

def scores(y_true, p):
    return {"auc": roc_auc_score(y_true, p), "accuracy": accuracy_score(y_true, p >= 0.5)}

## 4. Five-fold cross-validation: GAT vs BKT-only on the same students

In [ ]:
from scipy.stats import ttest_rel

cv = []
for k in range(5):
    train_rows = np.flatnonzero((fold >= 0) & (fold != k))
    val_rows = np.flatnonzero(fold == k)
    print(f"fold {k}: {len(train_rows):,} training answers, {len(val_rows):,} validation answers")
    model = train(train_rows)
    gat = scores(y[val_rows], predict(model, val_rows))
    bkt_only = scores(y[val_rows], p_bkt[val_rows])
    cv.append({"fold": k, "gat_auc": gat["auc"], "bkt_auc": bkt_only["auc"],
               "gat_accuracy": gat["accuracy"], "bkt_accuracy": bkt_only["accuracy"]})
    print(f"  GAT AUC {gat['auc']:.4f}   BKT AUC {bkt_only['auc']:.4f}")

cv = pd.DataFrame(cv)
t, p = ttest_rel(cv.gat_auc, cv.bkt_auc)
summary_cv = {"gat_auc_mean": cv.gat_auc.mean(), "gat_auc_sd": cv.gat_auc.std(),
              "bkt_auc_mean": cv.bkt_auc.mean(), "bkt_auc_sd": cv.bkt_auc.std(),
              "paired_t": t, "paired_p": p}
display(cv.round(4))
print({k: round(float(v), 4) for k, v in summary_cv.items()})

## 5. Final model on all training students, scored once on the held-out test students

In [ ]:
train_rows = np.flatnonzero(fold >= 0)
test_rows = np.flatnonzero(is_test)
final_model = train(train_rows)
p_gat_test = predict(final_model, test_rows)

majority = float(y[train_rows].mean() >= 0.5)
test_results = pd.DataFrame([
    {"model": "majority class", "auc": 0.5, "accuracy": accuracy_score(y[test_rows], np.full(len(test_rows), majority))},
    {"model": "BKT only", **scores(y[test_rows], p_bkt[test_rows])},
    {"model": "BKT + GAT (gat-v1)", **scores(y[test_rows], p_gat_test)},
]).round(4)
test_results

## 6. Export for numpy, and prove the numpy version gives the same answers

The service has no PyTorch. `gat_forward_numpy` below is the reference the service copies; it must match PyTorch on test answers.

In [ ]:
def export_weights(model):
    sd = {k: v.detach().cpu().numpy().astype(np.float32) for k, v in model.state_dict().items()}
    return {"W": sd["W.weight"].T, "a_dst": sd["a_dst"], "a_src": sd["a_src"],
            "fc1_w": sd["fc1.weight"].T, "fc1_b": sd["fc1.bias"],
            "fc2_w": sd["fc2.weight"].T, "fc2_b": sd["fc2.bias"]}

def gat_forward_numpy(w, x_t, x_n, mask, heads, hidden, slope):
    """x_t [B,F], x_n [B,K,F], mask [B,K] -> (probability [B], attention [B,1+K,H])."""
    B, k, _ = x_n.shape
    nodes = np.concatenate([x_t[:, None], x_n], 1)
    h = (nodes @ w["W"]).reshape(B, 1 + k, heads, hidden)
    score = (h[:, :1] * w["a_dst"]).sum(-1) + (h * w["a_src"]).sum(-1)
    score = np.where(score > 0, score, slope * score)
    present = np.concatenate([np.ones((B, 1), bool), mask], 1)
    score = np.where(present[..., None], score, -np.inf)
    score = score - score.max(1, keepdims=True)
    alpha = np.exp(score) / np.exp(score).sum(1, keepdims=True)
    z = (alpha[..., None] * h).sum(1).reshape(B, heads * hidden)
    z = np.where(z > 0, z, np.expm1(z))
    hidden1 = np.maximum(np.concatenate([z, x_t], -1) @ w["fc1_w"] + w["fc1_b"], 0)
    logit = (hidden1 @ w["fc2_w"] + w["fc2_b"])[:, 0]
    return 1 / (1 + np.exp(-logit)), alpha

weights = export_weights(final_model)
check = test_rows[:20_000]
p_np, _ = gat_forward_numpy(weights, X_t[check], X_n[check], M[check],
                            CONFIG["HEADS"], CONFIG["HIDDEN"], CONFIG["LEAKY_SLOPE"])
gap = np.abs(p_np - predict(final_model, check)).max()
print(f"max |numpy - torch| = {gap:.2e}")
assert gap < 1e-4, "numpy export does not reproduce the model"

## 7. Attention and the explanation rule

For answers on exercises that have prerequisites: how much attention goes to prerequisite edges, and which prerequisite the explanation rule would name (argmax of attention x (1 - mastery), attention averaged over heads).

In [ ]:
has_prereq = check[M[check].any(1)]
_, alpha = gat_forward_numpy(weights, X_t[has_prereq], X_n[has_prereq], M[has_prereq],
                             CONFIG["HEADS"], CONFIG["HIDDEN"], CONFIG["LEAKY_SLOPE"])
edge_attention = alpha[:, 1:].mean(-1)                         # [B, K], mean over heads
weakness = edge_attention * (1 - X_n[has_prereq][..., 0])
weakness[~M[has_prereq]] = -1
named = weakness.argmax(1)
print(f"answers with prerequisites: {len(has_prereq):,}")
print(f"mean attention on prerequisite edges (vs self-loop): {alpha[:, 1:].sum(1).mean():.3f}")
print(f"explanation names the least-mastered prerequisite in "
      f"{(named == np.where(M[has_prereq], X_n[has_prereq][..., 0], 9).argmin(1)).mean():.1%} of cases")

## 8. Save the model, the feature spec and a manifest

In [ ]:
import hashlib, platform, sklearn

model_path = OUT_DIR / f"{CONFIG['MODEL_VERSION']}.npz"
np.savez(model_path, **weights)
model_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()

feature_spec = {
    "model_version": CONFIG["MODEL_VERSION"],
    "features": FEATURES,
    "transforms": {"log_evidence": "log1p(count)", "log_depth": "log1p(depth)",
                   "log_prereq_count": "log1p(n_prerequisites)",
                   "mean_prereq_mastery": "mean over direct prerequisites, 1.0 when none"},
    "architecture": {"heads": CONFIG["HEADS"], "hidden": CONFIG["HIDDEN"],
                     "classifier": CONFIG["CLASSIFIER"], "leaky_slope": CONFIG["LEAKY_SLOPE"],
                     "self_loop": True, "attention_slot_0": "self", "max_prerequisites_trained": int(K)},
    "explanation": "weak prerequisite = argmax over prerequisite edges of mean-head attention x (1 - mastery)",
    "bkt_params": "junyi_processed/bkt/params.csv (notebook 03)",
    "sha256": model_sha,
    "test_results": test_results.to_dict(orient="records"),
    "cv": {k: float(v) for k, v in summary_cv.items()},
}
(OUT_DIR / "feature_spec.json").write_text(json.dumps(feature_spec, indent=2))
cv.to_csv(OUT_DIR / "cv_folds.csv", index=False)
test_results.to_csv(OUT_DIR / "test_results.csv", index=False)
print("model:", model_path.name, model_sha)

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "torch": torch.__version__, "scikit-learn": sklearn.__version__, "networkx": nx.__version__}

In [ ]:
import hashlib, json, platform
from datetime import datetime, timezone

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "versions": VERSIONS,
    "sha256": {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
               for f in sorted(OUT_DIR.glob("*")) if f.is_file() and f.name != "manifest.json"},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

In [ ]:
if STORAGE == "runtime":
    import shutil
    print("Download before stopping the runtime:", shutil.make_archive("/content/gat_outputs", "zip", OUT_DIR))
else:
    print("Outputs stored in", OUT_DIR)

## For the report and the service

- Report the CV table (mean and SD of AUC, paired t-test GAT vs BKT) and the test table against the 83.2% majority class.
- Copy `gat-v1.npz` and `feature_spec.json` to `curriculum-service` on Sat 11 (not into git); the service checks the SHA-256 before loading.
- Notebook 06 reuses this code for the ablations (no attention; similarity graph instead of prerequisites).

Before committing: **Clear all outputs**.